In [1]:
import logging
import pandas as pd

# ---------------------------------------------------------
# Configure logging
# ---------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# ---------------------------------------------------------
# Load raw CSV with proper exception handling
# ---------------------------------------------------------
def load_raw_csv(file_path: str) -> pd.DataFrame:
    """
    Load the raw CSV file with logging and safe exception handling.
    """
    logging.info(f"Attempting to load CSV file: {file_path}")

    try:
        df = pd.read_csv(file_path)
        logging.info(f"CSV loaded successfully. Shape: {df.shape}")
        return df

    except FileNotFoundError as e:
        logging.error(f"File not found: {file_path}")
        raise e

    except pd.errors.ParserError as e:
        logging.error(f"CSV parsing error: {e}")
        raise e

    except Exception as e:
        logging.error(f"Unexpected error occurred: {e}")
        raise e

# ---------------------------------------------------------
# Main pipeline entry point
# ---------------------------------------------------------
if __name__ == "__main__":
    logging.info("Starting data pipeline...")

    raw_file = "Metro_Interstate_Traffic_Volume.csv"

    df = load_raw_csv(raw_file)

    logging.info("Pipeline completed successfully.")



2026-09-23 23:08:17,213 - INFO - Starting data pipeline...
2026-09-23 23:08:17,214 - INFO - Attempting to load CSV file: Metro_Interstate_Traffic_Volume.csv
2026-09-23 23:08:17,340 - INFO - CSV loaded successfully. Shape: (48204, 9)
2026-09-23 23:08:17,341 - INFO - Pipeline completed successfully.


In [2]:
import logging
import pandas as pd

# ---------------------------------------------------------
# Configure logging
# ---------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# ---------------------------------------------------------
# Load raw CSV with proper exception handling
# ---------------------------------------------------------
def load_raw_csv(file_path: str) -> pd.DataFrame:
    """
    Load the raw CSV file with logging and safe exception handling.
    """
    logging.info(f"Attempting to load CSV file: {file_path}")

    try:
        df = pd.read_csv(file_path)
        logging.info(f"CSV loaded successfully. Shape: {df.shape}")
        return df

    except FileNotFoundError as e:
        logging.error(f"File not found: {file_path}")
        raise e

    except pd.errors.ParserError as e:
        logging.error(f"CSV parsing error: {e}")
        raise e

    except Exception as e:
        logging.error(f"Unexpected error occurred: {e}")
        raise e

# ---------------------------------------------------------
# Validate data schema
# ---------------------------------------------------------
def validate_schema(df: pd.DataFrame, expected_columns: list) -> None:
    """
    Validate that all expected columns are present in the dataset.
    """
    logging.info("Validating data schema...")

    missing_columns = [col for col in expected_columns if col not in df.columns]

    if missing_columns:
        logging.error(f"Schema validation failed. Missing columns: {missing_columns}")
        raise ValueError(f"Missing columns in dataset: {missing_columns}")

    logging.info("Schema validation successful. All expected columns are present.")

# ---------------------------------------------------------
# Main pipeline entry point
# ---------------------------------------------------------
if __name__ == "__main__":
    logging.info("Starting data pipeline...")

    raw_file = "Metro_Interstate_Traffic_Volume.csv"

    df = load_raw_csv(raw_file)

    expected_columns = [
        "holiday",
        "temp",
        "rain_1h",
        "snow_1h",
        "clouds_all",
        "weather_main",
        "weather_description",
        "date_time",
        "traffic_volume"
    ]

    validate_schema(df, expected_columns)

    logging.info("Pipeline completed successfully.")



2026-09-23 23:08:35,881 - INFO - Starting data pipeline...
2026-09-23 23:08:35,883 - INFO - Attempting to load CSV file: Metro_Interstate_Traffic_Volume.csv
2026-09-23 23:08:35,961 - INFO - CSV loaded successfully. Shape: (48204, 9)
2026-09-23 23:08:35,964 - INFO - Validating data schema...
2026-09-23 23:08:35,964 - INFO - Schema validation successful. All expected columns are present.
2026-09-23 23:08:35,965 - INFO - Pipeline completed successfully.


In [3]:
# ---------------------------------------------------------
# Clean the data
# ---------------------------------------------------------
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean the dataset by standardising categories, parsing dates,
    removing duplicates, detecting outliers, and imputing values.
    """
    logging.info("Starting data cleaning...")

    # -----------------------------------------------------
    # 1. Standardise inconsistent categorical values
    # -----------------------------------------------------
    logging.info("Standardising categorical values...")
    df["weather_main"] = df["weather_main"].str.strip().str.lower()
    df["weather_description"] = df["weather_description"].str.strip().str.lower()
    df["holiday"] = df["holiday"].str.strip().str.lower()

    # -----------------------------------------------------
    # 2. Parse and validate date/time fields
    # -----------------------------------------------------
    logging.info("Parsing date_time column...")
    try:
        df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce")
    except Exception as e:
        logging.error(f"Error parsing date_time: {e}")
        raise e

    # Remove rows where date_time could not be parsed
    invalid_dates = df["date_time"].isna().sum()
    if invalid_dates > 0:
        logging.warning(f"Removing {invalid_dates} rows with invalid date_time values.")
        df = df.dropna(subset=["date_time"])

    # -----------------------------------------------------
    # 3. Remove duplicate rows
    # -----------------------------------------------------
    logging.info("Removing duplicate rows...")
    before = len(df)
    df = df.drop_duplicates()
    after = len(df)
    logging.info(f"Removed {before - after} duplicate rows.")

    # -----------------------------------------------------
    # 4. Detect impossible values (outliers)
    # -----------------------------------------------------
    logging.info("Detecting impossible sensor values...")

    # Temperature: Kelvin cannot be 0
    impossible_temp = df[df["temp"] <= 0].shape[0]
    if impossible_temp > 0:
        logging.warning(f"Found {impossible_temp} impossible temperature values (<= 0 Kelvin).")

    # Rainfall: physically impossible > 9000 mm
    impossible_rain = df[df["rain_1h"] > 9000].shape[0]
    if impossible_rain > 0:
        logging.warning(f"Found {impossible_rain} impossible rainfall values (> 9000 mm).")

    # -----------------------------------------------------
    # 5. Impute outliers using median
    # -----------------------------------------------------
    logging.info("Imputing outliers using median...")

    temp_median = df[df["temp"] > 0]["temp"].median()
    df.loc[df["temp"] <= 0, "temp"] = temp_median

    rain_median = df[df["rain_1h"] <= 9000]["rain_1h"].median()
    df.loc[df["rain_1h"] > 9000, "rain_1h"] = rain_median

    logging.info("Data cleaning completed successfully.")
    return df

if __name__ == "__main__":
    logging.info("Starting data pipeline...")

    raw_file = "Metro_Interstate_Traffic_Volume.csv"

    df = load_raw_csv(raw_file)

    expected_columns = [
        "holiday",
        "temp",
        "rain_1h",
        "snow_1h",
        "clouds_all",
        "weather_main",
        "weather_description",
        "date_time",
        "traffic_volume"
    ]

    validate_schema(df, expected_columns)

    df = clean_data(df)
    logging.info(f"Data cleaning complete. Final shape: {df.shape}")

    logging.info("Pipeline completed successfully.")

2026-09-23 23:08:49,058 - INFO - Starting data pipeline...
2026-09-23 23:08:49,060 - INFO - Attempting to load CSV file: Metro_Interstate_Traffic_Volume.csv
2026-09-23 23:08:49,159 - INFO - CSV loaded successfully. Shape: (48204, 9)
2026-09-23 23:08:49,161 - INFO - Validating data schema...
2026-09-23 23:08:49,161 - INFO - Schema validation successful. All expected columns are present.
2026-09-23 23:08:49,162 - INFO - Starting data cleaning...
2026-09-23 23:08:49,162 - INFO - Standardising categorical values...
2026-09-23 23:08:49,195 - INFO - Parsing date_time column...
2026-09-23 23:08:49,216 - INFO - Removing duplicate rows...
2026-09-23 23:08:49,247 - INFO - Removed 17 duplicate rows.
2026-09-23 23:08:49,248 - INFO - Detecting impossible sensor values...
2026-09-23 23:08:49,251 - WARNING - Found 10 impossible temperature values (<= 0 Kelvin).
2026-09-23 23:08:49,253 - WARNING - Found 1 impossible rainfall values (> 9000 mm).
2026-09-23 23:08:49,254 - INFO - Imputing outliers using 

In [4]:
# ---------------------------------------------------------
# Clean the data with conditionals and loops (monthly imputation)
# ---------------------------------------------------------
def clean_data_with_monthly_imputation(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean the dataset using conditionals and loops.
    Impute outliers using monthly medians instead of global medians.
    """
    logging.info("Starting advanced cleaning with monthly imputation...")

    # Ensure date_time is parsed
    df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce")
    df = df.dropna(subset=["date_time"])

    # Extract month for group-by processing
    df["month"] = df["date_time"].dt.month

    # Loop through each month and apply monthly medians
    for month in df["month"].unique():
        logging.info(f"Processing month: {month}")

        # Subset for the month
        month_df = df[df["month"] == month]

        # -----------------------------------------------------
        # Temperature outliers (<= 0 Kelvin)
        # -----------------------------------------------------
        temp_outliers = month_df[month_df["temp"] <= 0].shape[0]
        if temp_outliers > 0:
            logging.warning(f"Month {month}: Found {temp_outliers} impossible temperature values.")

            # Monthly median (only valid temps)
            temp_median = month_df[month_df["temp"] > 0]["temp"].median()

            # Apply imputation using conditionals
            df.loc[(df["month"] == month) & (df["temp"] <= 0), "temp"] = temp_median

        # -----------------------------------------------------
        # Rainfall outliers (> 9000 mm)
        # -----------------------------------------------------
        rain_outliers = month_df[month_df["rain_1h"] > 9000].shape[0]
        if rain_outliers > 0:
            logging.warning(f"Month {month}: Found {rain_outliers} impossible rainfall values.")

            # Monthly median (only valid rainfall)
            rain_median = month_df[month_df["rain_1h"] <= 9000]["rain_1h"].median()

            # Apply imputation using conditionals
            df.loc[(df["month"] == month) & (df["rain_1h"] > 9000), "rain_1h"] = rain_median

    logging.info("Monthly imputation completed successfully.")

    # Remove helper column
    df = df.drop(columns=["month"])

    return df

df = clean_data(df)
df = clean_data_with_monthly_imputation(df)

logging.info(f"Advanced cleaning complete. Final shape: {df.shape}")

2026-09-23 23:21:25,788 - INFO - Starting data cleaning...
2026-09-23 23:21:25,789 - INFO - Standardising categorical values...
2026-09-23 23:21:25,855 - INFO - Parsing date_time column...
2026-09-23 23:21:25,884 - INFO - Removing duplicate rows...
2026-09-23 23:21:25,907 - INFO - Removed 0 duplicate rows.
2026-09-23 23:21:25,908 - INFO - Detecting impossible sensor values...
2026-09-23 23:21:25,913 - INFO - Imputing outliers using median...
2026-09-23 23:21:25,924 - INFO - Data cleaning completed successfully.
2026-09-23 23:21:25,926 - INFO - Starting advanced cleaning with monthly imputation...
2026-09-23 23:21:25,950 - INFO - Processing month: 10
2026-09-23 23:21:25,956 - INFO - Processing month: 11
2026-09-23 23:21:25,959 - INFO - Processing month: 12
2026-09-23 23:21:25,963 - INFO - Processing month: 1
2026-09-23 23:21:25,966 - INFO - Processing month: 2
2026-09-23 23:21:25,969 - INFO - Processing month: 3
2026-09-23 23:21:25,973 - INFO - Processing month: 4
2026-09-23 23:21:25,97